# Kafka + ETL Pipeline (Real-Time Data Engineering System)


## Why This Matters

You have built:
- ETL pipelines (batch)
- Kafka streams (real-time)

Now combine them:

👉 Real-time ingestion + transformation + storage

This is how modern data systems work.


## Pipeline Flow

Producer -> Kafka -> Consumer -> Transform -> PostgreSQL

👉 Real-time ETL system


## Producer (Event Generator)

```python
from kafka import KafkaProducer
import json
import time
import random

producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

events = ["click", "purchase", "login"]

while True:
    data = {
        "user_id": random.randint(1, 100),
        "event": random.choice(events),
        "amount": random.randint(100, 1000)
    }
    producer.send("events-topic", value=data)
    print("Sent:", data)
    time.sleep(1)
```


## Consumer + Transform + Load

```python
from kafka import KafkaConsumer
import psycopg2
import json

consumer = KafkaConsumer(
    "events-topic",
    bootstrap_servers='localhost:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    auto_offset_reset='earliest'
)

conn = psycopg2.connect(
    host="localhost",
    database="de_course",
    user="admin",
    password="admin"
)

cur = conn.cursor()

# Create table
cur.execute("""
CREATE TABLE IF NOT EXISTS events (
    user_id INT,
    event TEXT,
    amount INT
)
""")
conn.commit()

print("Processing stream...")

for message in consumer:
    data = message.value

    # Transform
    cleaned = {
        "user_id": data["user_id"],
        "event": data["event"].upper(),
        "amount": int(data["amount"])
    }

    # Load
    cur.execute("""
    INSERT INTO events (user_id, event, amount)
    VALUES (%s, %s, %s)
    """, (cleaned["user_id"], cleaned["event"], cleaned["amount"]))

    conn.commit()

    print("Inserted:", cleaned)
```


## Validate Data

```python
cur.execute("SELECT * FROM events LIMIT 10")
print(cur.fetchall())
```


## Real-Time Analytics

```python
cur.execute("""
SELECT event, COUNT(*) FROM events GROUP BY event
""")

print(cur.fetchall())
```


## Practice

1. Count events in real-time
2. Filter purchase events
3. Print high-value transactions


## Assignment

Upgrade pipeline:

- Add:
  - timestamp column
  - logging
  - retry mechanism

Bonus:
- Batch inserts instead of single insert


## Interview Questions

1. How do you build streaming ETL pipelines?
2. How does Kafka integrate with databases?
3. What are challenges in streaming systems?
4. How do you ensure data consistency?


## What You Just Built

This is HUGE:

👉 Real-time ingestion
👉 Transformation pipeline
👉 Database storage
👉 Streaming analytics

👉 This is exactly how fraud systems, tracking systems, and real-time dashboards work.


---

**Next:** Kafka Project (Portfolio Level)

You will:
- Build full streaming system
- Add multiple consumers
- Add analytics layer

👉 This becomes your top-tier project


**Reality check:** you can now build batch pipelines, streaming pipelines, and combined real-time ETL systems.


## Your tasks

- [ ] Create topic `events-topic`.
- [ ] Run event producer continuously.
- [ ] Run consumer pipeline and load into Postgres.
- [ ] Verify database updates in real-time with SQL checks.
